In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the flanger's four parameters. The top plot is its delay $M[n]$ (red),
which sweeps sinusoidally around $M_c$ (dashed). The bottom plot is the first
few teeth of its comb, with the lowest tooth $\frac{f_s}{M[n]}$ in bold. The
teeth linger low while the delay is long, and dart up whenever it is short.
The audio card plays the synthesizer pad through the flanger, mixed equally
with the dry pad. Feedback only changes the sound, not the plots: it sharpens
the teeth.

In [ ]:
# hide
# autorun
MC0, A0, FM0, G0 = 3.0, 0.9, 0.25, 0.7   # the chapter's flanger

PURPLE = "#6E3B87"
pad = pq.Audio.from_file("../assets/pad.wav").as_mono()
SR = pad.sample_rate
X = np.asarray(pad.samples).reshape(-1)
X = X / np.abs(X).max()
T_END = len(X) / SR
TP = np.linspace(0.0, T_END, 500)        # the plots' time grid
N_TEETH, F_TOP = 6, 5000.0


def delay_ms(mc, A, fm, t):
    return mc * (1 + A * np.sin(2 * np.pi * fm * t))


def flange(x, mc, A, fm, g, SR=SR):
    """y[n] = x[n] + g * y[n - M[n]], with a fractional delay M[n] read by
    linear interpolation. Computed in blocks no longer than the shortest
    delay, so every sample a block reads was finished in an earlier block."""
    n = np.arange(len(x))
    M = delay_ms(mc, A, fm, n / SR) / 1000 * SR
    k = M.astype(int)
    a = M - k
    B = max(1, int(M.min()))
    L = int(M.max()) + 2                 # zeros before n = 0
    z = np.zeros(L + len(x))
    for s in range(0, len(x), B):
        e = min(s + B, len(x))
        i = np.arange(L + s, L + e) - k[s:e]
        z[L + s:L + e] = x[s:e] + g * ((1 - a[s:e]) * z[i] + a[s:e] * z[i - 1])
    return z[L:]


def teeth(mc, A, fm, TP=TP, SR=SR):
    f1 = 1000 / delay_ms(mc, A, fm, TP)  # the lowest tooth, f_s / M[n]
    xs = np.concatenate([np.append(TP, np.nan)] * N_TEETH)
    ys = np.concatenate([np.append(j * f1, np.nan) for j in range(1, N_TEETH + 1)])
    return f1, xs, ys


def figure():
    fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14)
    fig.add_scatter(x=[0, T_END], y=[MC0, MC0], mode="lines",
                    line=dict(color=IRON, width=1.2, dash="dash"), row=1, col=1)
    fig.add_scatter(x=TP, y=delay_ms(MC0, A0, FM0, TP), mode="lines",
                    line=dict(color=RED, width=2), row=1, col=1)
    f1, xs, ys = teeth(MC0, A0, FM0)
    fig.add_scatter(x=xs, y=ys, mode="lines", opacity=0.35,
                    line=dict(color=PURPLE, width=1.2), row=2, col=1)
    fig.add_scatter(x=TP, y=f1, mode="lines", line=dict(color=PURPLE, width=2.4),
                    row=2, col=1)
    fig.update_xaxes(range=[0, T_END], fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[0, 20], title_text="Delay (ms)", fixedrange=True,
                     row=1, col=1)
    fig.update_xaxes(range=[0, T_END], title_text="Time (s)", fixedrange=True,
                     row=2, col=1)
    fig.update_yaxes(range=[0, F_TOP], title_text="Teeth (Hz)", fixedrange=True,
                     row=2, col=1)
    fig.update_layout(height=420)
    return fig


def controls(fig):
    mc = widgets.FloatSlider(description="Center delay $M_c$ (ms)", min=1.0,
                             max=10.0, step=0.1, value=MC0, readout_format=".1f")
    A = widgets.FloatSlider(description="Depth $A$", min=0.0, max=0.9,
                            step=0.01, value=A0)
    fm = widgets.FloatSlider(description="Rate $f_m$ (Hz)", min=0.05, max=2.0,
                             step=0.05, value=FM0)
    g = widgets.FloatSlider(description="Feedback $g$", min=0.0, max=0.9,
                            step=0.05, value=G0)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(mc, A, fm, g, delay_ms=delay_ms, teeth=teeth, TP=TP, SR=SR,
               readout=readout):
        f1, xs, ys = teeth(mc, A, fm)
        with fig.batch_update():
            fig.data[0].y = [mc, mc]
            fig.data[1].y = delay_ms(mc, A, fm, TP)
            fig.data[2].y = ys
            fig.data[3].y = f1
        lo, hi = mc * (1 - A), mc * (1 + A)
        readout.value = (
            f"<span style='font-size:0.9em'>delay sweeps {lo:.2f} to {hi:.2f} ms "
            f"({lo / 1000 * SR:.0f} to {hi / 1000 * SR:.0f} samples) every "
            f"{1 / fm:.1f} s &nbsp;·&nbsp; the lowest tooth sweeps "
            f"{1000 / hi:,.0f} to {1000 / lo:,.0f} Hz</span>")

    widgets.interactive_output(update, {"mc": mc, "A": A, "fm": fm, "g": g})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(X=X, flange=flange):
        wet = flange(X, mc.value, A.value, fm.value, g.value)
        y = 0.5 * X + 0.5 * wet            # a wet-to-dry ratio of 0 dB
        y = 0.5 * y / max(np.abs(y).max(), 1e-9)
        y[:441] *= np.linspace(0, 1, 441)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (mc, A, fm, g):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([mc, A, fm, g, readout, out, gate])

icm_plotly.show(figure, controls)